In [91]:
import pandas as pd
import numpy as np
import altair as alt

# Disable max rows limit to embed data inline (prevents creating external JSON files)
alt.data_transformers.enable('default', max_rows=None)

DataTransformerRegistry.enable('default')

In [128]:
# Read the road accidents with injuries data
accidents_with_injuries = pd.read_csv('../data/Road accidents with injuries (IT1,41_269_DF_DCIS_INCIDENTISTR1_1,1.0).csv')
accidents_with_injuries

,FREQ,Frequency,REF_AREA,Territory,DATA_TYPE,Indicator,ACCIDENT_LOCALIZATON,Localization of the accident,INTERSECTION,Intersection (DESC),...,Y_DEADLY_ACCIDENT,Deadly accident,HO_ROAD_ACCIDENT,Road accident hour,WEEK_DAY,Week day,MONTH,Month (DESC),TIME_PERIOD,Observation
0,A,Annual,ITC1,Piemonte,ROADACC,Road accidents with injuries,9,Total,9,Total,...,0,No,99,Total,9,Total,99,Total,2010,9473
1,A,Annual,ITC1,Piemonte,ROADACC,Road accidents with injuries,9,Total,9,Total,...,0,No,99,Total,9,Total,99,Total,2011,9376
2,A,Annual,ITC1,Piemonte,ROADACC,Road accidents with injuries,9,Total,9,Total,...,0,No,99,Total,9,Total,99,Total,2012,8542
3,A,Annual,ITC1,Piemonte,ROADACC,Road accidents with injuries,9,Total,9,Total,...,0,No,99,Total,9,Total,99,Total,2013,7710
4,A,Annual,ITC1,Piemonte,ROADACC,Road accidents with injuries,9,Total,9,Total,...,0,No,99,Total,9,Total,99,Total,2014,7804
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3941,A,Annual,ITG2,Sardegna,ROADACC,Road accidents with injuries,9,Total,9,Total,...,9,Total,99,Total,9,Total,99,Total,2020,2479
3942,A,Annual,ITG2,Sardegna,ROADACC,Road accidents with injuries,9,Total,9,Total,...,9,Total,99,Total,9,Total,99,Total,2021,3200
3943,A,Annual,ITG2,Sardegna,ROADACC,Road accidents with injuries,9,Total,9,Total,...,9,Total,99,Total,9,Total,99,Total,2022,3313
3944,A,Annual,ITG2,Sardegna,ROADACC,Road accidents with injuries,9,Total,9,Total,...,9,Total,99,Total,9,Total,99,Total,2023,3391


In [127]:
# Read both Italy regions provinces CSV files
pop_df1 = pd.read_csv('../data/Italy, regions, provinces (IT1,22_289_DF_DCIS_POPRES1_1,1.0).csv')
pop_df2 = pd.read_csv('../data/Italy, regions, provinces (IT1,22_289_DF_DCIS_POPRES1_1,1.0) (1).csv')

# Combine them into one population dataframe
population_df = pd.concat([pop_df1, pop_df2], ignore_index=True)
population_df = population_df.sort_values(['REF_AREA', 'TIME_PERIOD']).reset_index(drop=True)

population_df

,FREQ,Frequency,REF_AREA,Territory,DATA_TYPE,Indicator,SEX,Gender,AGE,Age (DESC),MARITAL_STATUS,Marital status,TIME_PERIOD,Observation,OBS_STATUS,Observation status
0,A,Annual,ITC1,Piemonte,JAN,Population on 1st January,9,Total,TOTAL,Total,99,Total,2019,4328565,NaN,NaN
1,A,Annual,ITC1,Piemonte,JAN,Population on 1st January,9,Total,TOTAL,Total,99,Total,2020,4311217,NaN,NaN
2,A,Annual,ITC1,Piemonte,JAN,Population on 1st January,9,Total,TOTAL,Total,99,Total,2021,4274945,NaN,NaN
3,A,Annual,ITC1,Piemonte,JAN,Population on 1st January,9,Total,TOTAL,Total,99,Total,2022,4256350,NaN,NaN
4,A,Annual,ITC1,Piemonte,JAN,Population on 1st January,9,Total,TOTAL,Total,99,Total,2023,4251351,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
149,A,Annual,ITG2,Sardegna,JAN,Population on 1st January,9,Total,TOTAL,Total,99,Total,2021,1590044,NaN,NaN
150,A,Annual,ITG2,Sardegna,JAN,Population on 1st January,9,Total,TOTAL,Total,99,Total,2022,1587413,NaN,NaN
151,A,Annual,ITG2,Sardegna,JAN,Population on 1st January,9,Total,TOTAL,Total,99,Total,2023,1578146,NaN,NaN
152,A,Annual,ITG2,Sardegna,JAN,Population on 1st January,9,Total,TOTAL,Total,99,Total,2024,1570453,NaN,NaN


In [94]:
# Prepare population data - get total population per region per year
pop_filtered = population_df[
    (population_df['Gender'] == 'Total') &
    (population_df['Age (DESC)'] == 'Total') &
    (population_df['Marital status'] == 'Total')
][['Territory', 'TIME_PERIOD', 'Observation']].copy()

pop_filtered = pop_filtered.rename(columns={'Observation': 'Population'})

# BACKFILL POPULATION DATA FOR 2010-2018
# Since population data is only available for 2019-2025, we backfill 2010-2018 
# using the 2019 population values for each region (assuming relatively stable population)
pop_2019 = pop_filtered[pop_filtered['TIME_PERIOD'] == 2019].copy()

backfilled_data = []
for year in range(2010, 2019):
    year_data = pop_2019.copy()
    year_data['TIME_PERIOD'] = year
    backfilled_data.append(year_data)

backfilled_df = pd.concat(backfilled_data, ignore_index=True)

# Combine backfilled data with original population data
pop_filtered = pd.concat([backfilled_df, pop_filtered], ignore_index=True).sort_values(['Territory', 'TIME_PERIOD']).reset_index(drop=True)

# Calculate total population for all of Italy by year
# EXCLUDE 'Trentino Alto Adige / Südtirol' to avoid double-counting
# (its population is already included in Provincia Autonoma Bolzano and Provincia Autonoma Trento)
italy_total_pop = pop_filtered[
    pop_filtered['Territory'] != 'Trentino Alto Adige / Südtirol'
].groupby('TIME_PERIOD')['Population'].sum().reset_index()
italy_total_pop['Territory'] = 'All Regions'

# Combine regional and total population
population_data = pd.concat([pop_filtered, italy_total_pop], ignore_index=True)

In [116]:
# Read the mobility and road conditions dataset
mobility_df = pd.read_csv('../data/Mobility and road conditions (IT1,82_87_DF_DCCV_AVQ_FAMIGLIE_104,1.0).csv')

In [96]:
# Read the usual way of getting to work dataset
work_commute_df = pd.read_csv('../data/Usual way of getting to work - regions and type of municipality (IT1,83_63_DF_DCCV_AVQ_PERSONE_170,1.0).csv')

In [97]:
# Filter out autonomous provinces (Provincia Autonoma) from accidents data
accidents_filtered = accidents_with_injuries[
    ~accidents_with_injuries['Territory'].str.contains('Provincia Autonoma', na=False)
].copy()

In [130]:
# Prepare accidents data - filter for total values to avoid double counting
accidents_data = accidents_filtered[
    (accidents_filtered['Localization of the accident'] == 'Total') &
    (accidents_filtered['Intersection (DESC)'] == 'Total') &
    (accidents_filtered['Road accident type'] == 'Total')
].copy()

# Group by region and year
accidents_by_region_year = accidents_data.groupby([
    'Territory', 
    'TIME_PERIOD'
])['Observation'].sum().reset_index()

accidents_by_region_year = accidents_by_region_year.rename(columns={'Observation': 'Total_Accidents'})

# Calculate accidents for all of Italy by year
# EXCLUDE 'Trentino Alto Adige / Südtirol' to avoid double-counting
# (its accidents are already included in the filtered data which excludes Provincia Autonoma)
accidents_all_italy = accidents_by_region_year[
    accidents_by_region_year['Territory'] != 'Trentino Alto Adige / Südtirol'
].groupby('TIME_PERIOD')['Total_Accidents'].sum().reset_index()
accidents_all_italy['Territory'] = 'All Regions'

# Combine regional and total accidents
accidents_by_region_year = pd.concat([accidents_all_italy, accidents_by_region_year], ignore_index=True)

In [99]:
# Merge accidents with population data to calculate per 100,000 inhabitants
accidents_with_pop = accidents_by_region_year.merge(
    population_data,
    on=['Territory', 'TIME_PERIOD'],
    how='left'
)

# Calculate accidents per 100,000 inhabitants
accidents_with_pop['Accidents_Per_100k'] = (accidents_with_pop['Total_Accidents'] / accidents_with_pop['Population']) * 100000

# Display the result
accidents_with_pop

,Territory,TIME_PERIOD,Total_Accidents,Population,Accidents_Per_100k
0,"'Valle d""'Aosta / Vallée d""'Aoste'",2010,740,125653,588.923464
1,"'Valle d""'Aosta / Vallée d""'Aoste'",2011,598,125653,475.913826
2,"'Valle d""'Aosta / Vallée d""'Aoste'",2012,590,125653,469.547086
3,"'Valle d""'Aosta / Vallée d""'Aoste'",2013,630,125653,501.380787
4,"'Valle d""'Aosta / Vallée d""'Aoste'",2014,590,125653,469.547086
...,...,...,...,...,...
295,Veneto,2020,19678,4879133,403.309358
296,Veneto,2021,24806,4869830,509.381231
297,Veneto,2022,26440,4847745,545.408226
298,Veneto,2023,25548,4849553,526.811440


In [100]:
# Explore mobility_df indicators
print("Mobility indicators:")
print(mobility_df['Indicator'].unique())

Mobility indicators:
['Households declaring to live in an area where some problems are present: difficult parking: very much'
 'Households declaring to live in an area where some problems are present: difficult parking: very much and quite'
 'Households declaring to live in an area where some problems are present: difficulties of links with public transport means: very much'
 'Households declaring to live in an area where some problems are present: difficulties of links with public transport means: very much and quite'
 'Households declaring to live in an area where some problems are present: traffic: very much'
 'Households declaring to live in an area where some problems are present: traffic: very much and quite'
 'Households declaring to live in an area where some problems are present: poor street lighting: very much'
 'Households declaring to live in an area where some problems are present: poor street lighting: very much and quite'
 'Households declaring to live in an area where s

In [101]:
# Explore work_commute_df structure
print("Work commute indicators:")
print(work_commute_df['Indicator'].unique())
print("\nColumns:", work_commute_df.columns.tolist())
print("\nAge groups:", work_commute_df['Age (DESC)'].unique())

Work commute indicators:
['Employed aged 15 years and over who leave home to go to the work by means of trasport used and time spent: by foot'
 'Employed aged 15 years and over who leave home to go to the work by means of trasport used and time spent: they use means of transport'
 'Employed aged 15 years and over who leave home to go to the work by means of trasport used and time spent: train'
 "'Employed aged 15 years and over who leave home to go to the work by means of trasport used and time spent: tram"
 'Employed aged 15 years and over who leave home to go to the work by means of trasport used and time spent: metro'
 'Employed aged 15 years and over who leave home to go to the work by means of trasport used and time spent: coach'
 'Employed aged 15 years and over who leave home to go to the work by means of trasport used and time spent: bus company'
 'Employed aged 15 years and over who leave home to go to the work by means of trasport used and time spent: private car as driver'
 

In [139]:
# Pivot mobility data - each indicator becomes a column
mobility_pivot = mobility_df.pivot_table(
    index=['Territory', 'TIME_PERIOD'],
    columns='DATA_TYPE',
    values='Observation',
    aggfunc='first'
).reset_index()

# Rename columns with prefix for clarity
mobility_pivot.columns = ['Territory', 'TIME_PERIOD'] + [f'mobility_{col}' for col in mobility_pivot.columns[2:]]

mobility_pivot

,Territory,TIME_PERIOD,mobility_HOUS_DIFFPARK_V,mobility_HOUS_DIFFPARK_VQ,mobility_HOUS_DIFF_TRANS_V,mobility_HOUS_DIFF_TRANS_VQ,mobility_HOUS_PROAD_V,mobility_HOUS_PROAD_VQ,mobility_HOUS_STRLIGHT_V,mobility_HOUS_STRLIGHT_VQ,mobility_HOUS_TRAFFIC_V,mobility_HOUS_TRAFFIC_VQ
0,"'Valle d""'Aosta / Vallée d""'Aoste'",2010,13.3,34.2,7.6,23.6,12.7,38.9,4.3,18.3,7.5,26.2
1,"'Valle d""'Aosta / Vallée d""'Aoste'",2011,16.6,30.0,5.2,18.6,8.5,30.5,4.5,20.7,9.3,24.9
2,"'Valle d""'Aosta / Vallée d""'Aoste'",2012,11.0,25.2,6.5,20.2,8.9,28.6,4.5,18.2,4.9,20.4
3,"'Valle d""'Aosta / Vallée d""'Aoste'",2013,10.4,24.0,7.1,21.5,10.7,30.8,2.8,15.5,4.9,18.7
4,"'Valle d""'Aosta / Vallée d""'Aoste'",2014,10.9,22.9,9.9,25.7,9.4,30.7,5.2,19.8,5.1,18.6
...,...,...,...,...,...,...,...,...,...,...,...,...
340,Veneto,2020,8.2,24.7,9.3,26.0,9.8,33.1,6.7,21.6,10.2,35.0
341,Veneto,2021,6.9,22.0,8.4,25.4,12.3,38.4,5.3,21.2,8.2,33.4
342,Veneto,2022,9.6,24.4,9.4,26.4,13.1,38.5,6.3,24.2,10.7,37.0
343,Veneto,2023,10.4,28.4,10.9,31.9,13.2,39.6,7.5,27.3,11.2,38.3


In [140]:
# Filter work commute data for "15 years and over" age group before pivoting
work_commute_filtered = work_commute_df[work_commute_df['Age (DESC)'] == '15 years and over'].copy()

# Pivot work commute data - each indicator becomes a column
work_commute_pivot = work_commute_filtered.pivot_table(
    index=['Territory', 'TIME_PERIOD'],
    columns='DATA_TYPE',
    values='Observation',
    aggfunc='first'
).reset_index()

# Rename columns with prefix for clarity
work_commute_pivot.columns = ['Territory', 'TIME_PERIOD'] + [f'commute_{col}' for col in work_commute_pivot.columns[2:]]

# Convert TIME_PERIOD to int to match other dataframes
work_commute_pivot['TIME_PERIOD'] = work_commute_pivot['TIME_PERIOD'].astype(int)

work_commute_pivot

,Territory,TIME_PERIOD,commute_15_EMPL_MOVT_15,commute_15_EMPL_MOVT_31,commute_15_EMPMOV_BICYC,commute_15_EMPMOV_BUSC,commute_15_EMPMOV_COACH,commute_15_EMPMOV_FOOT,commute_15_EMPMOV_MEANS,commute_15_EMPMOV_METRO,commute_15_EMPMOV_PCAR,commute_15_EMPMOV_PPASS,commute_15_EMPMOV_TRAIN
0,"'Valle d""'Aosta / Vallée d""'Aoste'",2010,50.1,11.1,0.8,1.4,1.8,14.4,85.1,0.2,76.5,6.9,1.5
1,"'Valle d""'Aosta / Vallée d""'Aoste'",2011,49.5,11.3,3.4,1.2,1,18.2,81.8,0.3,72.3,3.6,3.2
2,"'Valle d""'Aosta / Vallée d""'Aoste'",2012,58.6,8,2.3,0.7,3.7,15,84.2,0.2,77,2.2,2.6
3,"'Valle d""'Aosta / Vallée d""'Aoste'",2013,55.2,7,1.6,1.1,2.4,20.2,78,0.5,69.2,4.1,2.3
4,"'Valle d""'Aosta / Vallée d""'Aoste'",2014,55.6,8.5,1.7,0.7,3.3,20.3,79.4,0,67.8,5.2,2.4
...,...,...,...,...,...,...,...,...,...,...,...,...,...
340,Veneto,2020,41.6,14.1,4.6,0,1.1,6.4,93.6,0,79.5,3.7,1.6
341,Veneto,2021,39.3,15.2,6.7,0.3,1.5,7.3,92.7,0,77.9,3.8,1.6
342,Veneto,2022,40.6,12.9,6.3,0.1,1.3,7,93,0.1,77.2,4,1.5
343,Veneto,2023,42.2,14,6.3,0.1,1.9,8.9,91.1,0.3,75,4,1.8


In [104]:
# Merge mobility data into accidents_with_pop
accidents_with_pop = accidents_with_pop.merge(
    mobility_pivot,
    on=['Territory', 'TIME_PERIOD'],
    how='left'
)

# Merge work commute data into accidents_with_pop
accidents_with_pop = accidents_with_pop.merge(
    work_commute_pivot,
    on=['Territory', 'TIME_PERIOD'],
    how='left'
)

# Display the merged dataframe
print(f"Shape: {accidents_with_pop.shape}")
accidents_with_pop.head()

Shape: (300, 26)


,Territory,TIME_PERIOD,Total_Accidents,Population,Accidents_Per_100k,mobility_HOUS_DIFFPARK_V,mobility_HOUS_DIFFPARK_VQ,mobility_HOUS_DIFF_TRANS_V,mobility_HOUS_DIFF_TRANS_VQ,mobility_HOUS_PROAD_V,...,commute_15_EMPL_MOVT_31,commute_15_EMPMOV_BICYC,commute_15_EMPMOV_BUSC,commute_15_EMPMOV_COACH,commute_15_EMPMOV_FOOT,commute_15_EMPMOV_MEANS,commute_15_EMPMOV_METRO,commute_15_EMPMOV_PCAR,commute_15_EMPMOV_PPASS,commute_15_EMPMOV_TRAIN
0,"'Valle d""'Aosta / Vallée d""'Aoste'",2010,740,125653,588.923464,13.3,34.2,7.6,23.6,12.7,...,11.1,0.8,1.4,1.8,14.4,85.1,0.2,76.5,6.9,1.5
1,"'Valle d""'Aosta / Vallée d""'Aoste'",2011,598,125653,475.913826,16.6,30.0,5.2,18.6,8.5,...,11.3,3.4,1.2,1,18.2,81.8,0.3,72.3,3.6,3.2
2,"'Valle d""'Aosta / Vallée d""'Aoste'",2012,590,125653,469.547086,11.0,25.2,6.5,20.2,8.9,...,8,2.3,0.7,3.7,15,84.2,0.2,77,2.2,2.6
3,"'Valle d""'Aosta / Vallée d""'Aoste'",2013,630,125653,501.380787,10.4,24.0,7.1,21.5,10.7,...,7,1.6,1.1,2.4,20.2,78,0.5,69.2,4.1,2.3
4,"'Valle d""'Aosta / Vallée d""'Aoste'",2014,590,125653,469.547086,10.9,22.9,9.9,25.7,9.4,...,8.5,1.7,0.7,3.3,20.3,79.4,0,67.8,5.2,2.4


In [105]:
# Check if there are more descriptive names in the mobility data
print("MOBILITY DATA - Checking for descriptive columns:")
print("\nColumns:", mobility_df.columns.tolist())
print("\nSample rows with DATA_TYPE and Indicator:")
print(mobility_df[['DATA_TYPE', 'Indicator']].drop_duplicates().sort_values('DATA_TYPE'))

MOBILITY DATA - Checking for descriptive columns:

Columns: ['FREQ', 'Frequency', 'REF_AREA', 'Territory', 'DATA_TYPE', 'Indicator', 'MEASURE', 'Measure (DESC)', 'TIME_PERIOD', 'Observation']

Sample rows with DATA_TYPE and Indicator:
              DATA_TYPE                                          Indicator
0       HOUS_DIFFPARK_V  Households declaring to live in an area where ...
15     HOUS_DIFFPARK_VQ  Households declaring to live in an area where ...
30    HOUS_DIFF_TRANS_V  Households declaring to live in an area where ...
45   HOUS_DIFF_TRANS_VQ  Households declaring to live in an area where ...
120        HOUS_PROAD_V  Households declaring to live in an area where ...
135       HOUS_PROAD_VQ  Households declaring to live in an area where ...
90      HOUS_STRLIGHT_V  Households declaring to live in an area where ...
105    HOUS_STRLIGHT_VQ  Households declaring to live in an area where ...
60       HOUS_TRAFFIC_V  Households declaring to live in an area where ...
75      HOUS_TR

In [106]:
# Check if there are more descriptive names in the work commute data
print("WORK COMMUTE DATA - Checking for descriptive columns:")
print("\nColumns:", work_commute_df.columns.tolist())
print("\nSample rows with DATA_TYPE and Indicator:")
print(work_commute_df[['DATA_TYPE', 'Indicator']].drop_duplicates().sort_values('DATA_TYPE'))

WORK COMMUTE DATA - Checking for descriptive columns:

Columns: ['FREQ', 'Frequency', 'REF_AREA', 'Territory', 'DATA_TYPE', 'Indicator', 'MEASURE', 'Measure (DESC)', 'AGE', 'Age (DESC)', 'TIME_PERIOD', 'Observation']

Sample rows with DATA_TYPE and Indicator:
           DATA_TYPE                                          Indicator
165  15_EMPL_MOVT_15  Employed aged 15 years and over who leave home...
180  15_EMPL_MOVT_31  Employed aged 15 years and over who leave home...
150  15_EMPMOV_BICYC  Employed aged 15 years and over who leave home...
45     15_EMPMOV_BUS  'Employed aged 15 years and over who leave hom...
90    15_EMPMOV_BUSC  Employed aged 15 years and over who leave home...
75   15_EMPMOV_COACH  Employed aged 15 years and over who leave home...
0     15_EMPMOV_FOOT  Employed aged 15 years and over who leave home...
15   15_EMPMOV_MEANS  Employed aged 15 years and over who leave home...
60   15_EMPMOV_METRO  Employed aged 15 years and over who leave home...
135  15_EMPMOV_MOTOR

In [117]:
# Get full indicator text for mobility data
print("MOBILITY INDICATORS (full text):")
mobility_mapping = mobility_df[['DATA_TYPE', 'Indicator']].drop_duplicates().set_index('DATA_TYPE')['Indicator'].to_dict()
for code, description in sorted(mobility_mapping.items()):
    print(f"\n{code}:")
    print(f"  {description}")

MOBILITY INDICATORS (full text):

HOUS_DIFFPARK_V:
  Households declaring to live in an area where some problems are present: difficult parking: very much

HOUS_DIFFPARK_VQ:
  Households declaring to live in an area where some problems are present: difficult parking: very much and quite

HOUS_DIFF_TRANS_V:
  Households declaring to live in an area where some problems are present: difficulties of links with public transport means: very much

HOUS_DIFF_TRANS_VQ:
  Households declaring to live in an area where some problems are present: difficulties of links with public transport means: very much and quite

HOUS_PROAD_V:
  Households declaring to live in an area where some problems are present: poor road conditions: very much

HOUS_PROAD_VQ:
  Households declaring to live in an area where some problems are present: poor road conditions: very much and quite

HOUS_STRLIGHT_V:
  Households declaring to live in an area where some problems are present: poor street lighting: very much

HOUS_STR

In [108]:
# Get full indicator text for work commute data
print("WORK COMMUTE INDICATORS (full text):")
commute_mapping = work_commute_df[['DATA_TYPE', 'Indicator']].drop_duplicates().set_index('DATA_TYPE')['Indicator'].to_dict()
for code, description in sorted(commute_mapping.items()):
    print(f"\n{code}:")
    print(f"  {description}")

WORK COMMUTE INDICATORS (full text):

15_EMPL_MOVT_15:
  Employed aged 15 years and over who leave home to go to the work by means of trasport used and time spent: until 15 minutes

15_EMPL_MOVT_31:
  Employed aged 15 years and over who leave home to go to the work by means of trasport used and time spent: 31 minutes and over

15_EMPMOV_BICYC:
  Employed aged 15 years and over who leave home to go to the work by means of trasport used and time spent: bicycle

15_EMPMOV_BUS:
  'Employed aged 15 years and over who leave home to go to the work by means of trasport used and time spent: tram

15_EMPMOV_BUSC:
  Employed aged 15 years and over who leave home to go to the work by means of trasport used and time spent: bus company

15_EMPMOV_COACH:
  Employed aged 15 years and over who leave home to go to the work by means of trasport used and time spent: coach

15_EMPMOV_FOOT:
  Employed aged 15 years and over who leave home to go to the work by means of trasport used and time spent: by foot



## Using More Descriptive Column Names

We can create more readable column names by using the Indicator descriptions. Here are shortened versions:

In [119]:
# Create descriptive name mappings
mobility_names = {
    'HOUS_DIFFPARK_V': 'Difficult_Parking_V',
    'HOUS_DIFFPARK_VQ': 'Difficult_Parking_VQ',
    'HOUS_DIFF_TRANS_V': 'Poor_Public_Transport_V',
    'HOUS_DIFF_TRANS_VQ': 'Poor_Public_Transport_VQ',
    'HOUS_PROAD_V': 'Poor_Road_Conditions_V',
    'HOUS_PROAD_VQ': 'Poor_Road_Conditions_VQ',
    'HOUS_STRLIGHT_V': 'Poor_Street_Lighting_V',
    'HOUS_STRLIGHT_VQ': 'Poor_Street_Lighting_VQ',
    'HOUS_TRAFFIC_V': 'Traffic_Problems_V',
    'HOUS_TRAFFIC_VQ': 'Traffic_Problems_VQ'
}

commute_names = {
    '15_EMPL_MOVT_15': 'Commute_Under_15min',
    '15_EMPL_MOVT_31': 'Commute_Over_31min',
    '15_EMPMOV_BICYC': 'Commute_Bicycle',
    '15_EMPMOV_BUSC': 'Commute_Bus',
    '15_EMPMOV_COACH': 'Commute_Coach',
    '15_EMPMOV_FOOT': 'Commute_Walk',
    '15_EMPMOV_MEANS': 'Commute_Any_Transport',
    '15_EMPMOV_METRO': 'Commute_Metro',
    '15_EMPMOV_PCAR': 'Commute_Car_Driver',
    '15_EMPMOV_PPASS': 'Commute_Car_Passenger',
    '15_EMPMOV_TRAIN': 'Commute_Train'
}

In [142]:
# Pivot mobility data with descriptive names
mobility_pivot_v2 = mobility_df.pivot_table(
    index=['Territory', 'TIME_PERIOD'],
    columns='DATA_TYPE',
    values='Observation',
    aggfunc='first'
).reset_index()

# Rename columns using the descriptive mapping
mobility_pivot_v2.columns = ['Territory', 'TIME_PERIOD'] + [
    mobility_names.get(col, col) for col in mobility_pivot_v2.columns[2:]
]

# Rename 'Italy' to 'All Regions' to match accidents data
mobility_pivot_v2['Territory'] = mobility_pivot_v2['Territory'].replace('Italy', 'All Regions')

mobility_pivot_v2.head()


,Territory,TIME_PERIOD,Difficult_Parking_V,Difficult_Parking_VQ,Poor_Public_Transport_V,Poor_Public_Transport_VQ,Poor_Road_Conditions_V,Poor_Road_Conditions_VQ,Poor_Street_Lighting_V,Poor_Street_Lighting_VQ,Traffic_Problems_V,Traffic_Problems_VQ
0,"'Valle d""'Aosta / Vallée d""'Aoste'",2010,13.3,34.2,7.6,23.6,12.7,38.9,4.3,18.3,7.5,26.2
1,"'Valle d""'Aosta / Vallée d""'Aoste'",2011,16.6,30.0,5.2,18.6,8.5,30.5,4.5,20.7,9.3,24.9
2,"'Valle d""'Aosta / Vallée d""'Aoste'",2012,11.0,25.2,6.5,20.2,8.9,28.6,4.5,18.2,4.9,20.4
3,"'Valle d""'Aosta / Vallée d""'Aoste'",2013,10.4,24.0,7.1,21.5,10.7,30.8,2.8,15.5,4.9,18.7
4,"'Valle d""'Aosta / Vallée d""'Aoste'",2014,10.9,22.9,9.9,25.7,9.4,30.7,5.2,19.8,5.1,18.6


In [143]:
# Pivot work commute data with descriptive names
work_commute_pivot_v2 = work_commute_filtered.pivot_table(
    index=['Territory', 'TIME_PERIOD'],
    columns='DATA_TYPE',
    values='Observation',
    aggfunc='first'
).reset_index()

# Rename columns using the descriptive mapping
work_commute_pivot_v2.columns = ['Territory', 'TIME_PERIOD'] + [
    commute_names.get(col, col) for col in work_commute_pivot_v2.columns[2:]
]

# Convert TIME_PERIOD to int to match other dataframes
work_commute_pivot_v2['TIME_PERIOD'] = work_commute_pivot_v2['TIME_PERIOD'].astype(int)

# Rename 'Italy' to 'All Regions' to match accidents data
work_commute_pivot_v2['Territory'] = work_commute_pivot_v2['Territory'].replace('Italy', 'All Regions')

work_commute_pivot_v2.head()


,Territory,TIME_PERIOD,Commute_Under_15min,Commute_Over_31min,Commute_Bicycle,Commute_Bus,Commute_Coach,Commute_Walk,Commute_Any_Transport,Commute_Metro,Commute_Car_Driver,Commute_Car_Passenger,Commute_Train
0,"'Valle d""'Aosta / Vallée d""'Aoste'",2010,50.1,11.1,0.8,1.4,1.8,14.4,85.1,0.2,76.5,6.9,1.5
1,"'Valle d""'Aosta / Vallée d""'Aoste'",2011,49.5,11.3,3.4,1.2,1,18.2,81.8,0.3,72.3,3.6,3.2
2,"'Valle d""'Aosta / Vallée d""'Aoste'",2012,58.6,8,2.3,0.7,3.7,15,84.2,0.2,77,2.2,2.6
3,"'Valle d""'Aosta / Vallée d""'Aoste'",2013,55.2,7,1.6,1.1,2.4,20.2,78,0.5,69.2,4.1,2.3
4,"'Valle d""'Aosta / Vallée d""'Aoste'",2014,55.6,8.5,1.7,0.7,3.3,20.3,79.4,0,67.8,5.2,2.4


In [144]:
# Create the final dataframe with descriptive column names
# Start fresh by recreating accidents_with_pop from cell 9
accidents_with_pop_v2 = accidents_by_region_year.merge(
    population_data,
    on=['Territory', 'TIME_PERIOD'],
    how='left'
)
accidents_with_pop_v2['Accidents_Per_100k'] = (accidents_with_pop_v2['Total_Accidents'] / accidents_with_pop_v2['Population']) * 100000

# Merge with descriptive mobility data
accidents_with_pop_v2 = accidents_with_pop_v2.merge(
    mobility_pivot_v2,
    on=['Territory', 'TIME_PERIOD'],
    how='left'
)

# Merge with descriptive work commute data
accidents_with_pop_v2 = accidents_with_pop_v2.merge(
    work_commute_pivot_v2,
    on=['Territory', 'TIME_PERIOD'],
    how='left'
)

print(f"Shape: {accidents_with_pop_v2.shape}")
print(f"\nColumns: {accidents_with_pop_v2.columns.tolist()}")
accidents_with_pop_v2.head()

Shape: (315, 26)

Columns: ['TIME_PERIOD', 'Total_Accidents', 'Territory', 'Population', 'Accidents_Per_100k', 'Difficult_Parking_V', 'Difficult_Parking_VQ', 'Poor_Public_Transport_V', 'Poor_Public_Transport_VQ', 'Poor_Road_Conditions_V', 'Poor_Road_Conditions_VQ', 'Poor_Street_Lighting_V', 'Poor_Street_Lighting_VQ', 'Traffic_Problems_V', 'Traffic_Problems_VQ', 'Commute_Under_15min', 'Commute_Over_31min', 'Commute_Bicycle', 'Commute_Bus', 'Commute_Coach', 'Commute_Walk', 'Commute_Any_Transport', 'Commute_Metro', 'Commute_Car_Driver', 'Commute_Car_Passenger', 'Commute_Train']


,TIME_PERIOD,Total_Accidents,Territory,Population,Accidents_Per_100k,Difficult_Parking_V,Difficult_Parking_VQ,Poor_Public_Transport_V,Poor_Public_Transport_VQ,Poor_Road_Conditions_V,...,Commute_Over_31min,Commute_Bicycle,Commute_Bus,Commute_Coach,Commute_Walk,Commute_Any_Transport,Commute_Metro,Commute_Car_Driver,Commute_Car_Passenger,Commute_Train
0,2010,420754,All Regions,59816673,703.405888,18.7,39.6,10.3,29.5,23.0,...,16.4,3.3,0.6,1.8,10.7,88.7,2.4,70.8,5.4,3.1
1,2011,405294,All Regions,59816673,677.560251,17.9,38.0,9.8,28.6,21.7,...,16.7,3.1,0.5,2.2,11.8,87.7,2.8,70.1,5.5,2.8
2,2012,369928,All Regions,59816673,618.436268,14.8,35.8,9.9,28.8,17.9,...,15.4,3.9,0.5,2,11.5,87.9,2.9,69.5,5,3.1
3,2013,356982,All Regions,59816673,596.793473,15.8,37.2,10.5,31.3,22.6,...,15.9,3.7,0.7,1.9,11.4,88,3.4,69,5.4,3.7
4,2014,348058,All Regions,59816673,581.874555,14.9,35.2,10.2,30.7,21.2,...,14.9,4.2,0.5,1.9,11.1,88.1,3.2,68.3,5.2,3.4


In [145]:
from scipy import stats
import numpy as np

# Prepare data - remove rows with missing values
data_for_regression = accidents_with_pop_v2.dropna()

# Get list of x-axis variables (mobility and commute columns)
x_variables = [
    'Difficult_Parking_V', 'Difficult_Parking_VQ',
    'Poor_Public_Transport_V', 'Poor_Public_Transport_VQ',
    'Poor_Road_Conditions_V', 'Poor_Road_Conditions_VQ',
    'Poor_Street_Lighting_V', 'Poor_Street_Lighting_VQ',
    'Traffic_Problems_V', 'Traffic_Problems_VQ',
    'Commute_Under_15min', 'Commute_Over_31min',
    'Commute_Bicycle', 'Commute_Bus', 'Commute_Coach',
    'Commute_Walk', 'Commute_Any_Transport', 'Commute_Metro',
    'Commute_Car_Driver', 'Commute_Car_Passenger', 'Commute_Train'
]

# Calculate regression statistics for each variable and year (cross-sectional)
# EXCLUDE 'All Regions' from cross-sectional analysis (only compare individual regions)
regression_data_cs = []

for year in data_for_regression['TIME_PERIOD'].unique():
    year_data = data_for_regression[
        (data_for_regression['TIME_PERIOD'] == year) & 
        (data_for_regression['Territory'] != 'All Regions')
    ]
    
    for var in x_variables:
        # Get clean data (no NaN values) and convert to numeric
        mask = year_data[[var, 'Accidents_Per_100k']].notna().all(axis=1)
        x_vals = pd.to_numeric(year_data.loc[mask, var], errors='coerce')
        y_vals = pd.to_numeric(year_data.loc[mask, 'Accidents_Per_100k'], errors='coerce')
        
        # Remove any NaN created during conversion
        valid_mask = x_vals.notna() & y_vals.notna()
        x_vals = x_vals[valid_mask].values
        y_vals = y_vals[valid_mask].values
        
        if len(x_vals) > 2:  # Need at least 3 points for regression
            # Calculate regression
            slope, intercept, r_value, p_value, std_err = stats.linregress(x_vals, y_vals)
            
            regression_data_cs.append({
                'TIME_PERIOD': year,
                'Variable': var,
                'slope': slope,
                'intercept': intercept,
                'r_squared': r_value ** 2,
                'p_value': p_value,
                'regression_type': 'cross_sectional'
            })

# Calculate regression statistics for each variable and region (time-series)
regression_data_ts = []

for region in data_for_regression['Territory'].unique():
    region_data = data_for_regression[data_for_regression['Territory'] == region]
    
    for var in x_variables:
        # Get clean data (no NaN values) and convert to numeric
        mask = region_data[[var, 'Accidents_Per_100k']].notna().all(axis=1)
        x_vals = pd.to_numeric(region_data.loc[mask, var], errors='coerce')
        y_vals = pd.to_numeric(region_data.loc[mask, 'Accidents_Per_100k'], errors='coerce')
        
        # Remove any NaN created during conversion
        valid_mask = x_vals.notna() & y_vals.notna()
        x_vals = x_vals[valid_mask].values
        y_vals = y_vals[valid_mask].values
        
        if len(x_vals) > 2:  # Need at least 3 points for regression
            # Calculate regression
            slope, intercept, r_value, p_value, std_err = stats.linregress(x_vals, y_vals)
            
            regression_data_ts.append({
                'Territory': region,
                'Variable': var,
                'slope': slope,
                'intercept': intercept,
                'r_squared': r_value ** 2,
                'p_value': p_value,
                'regression_type': 'time_series'
            })

regression_df_cs = pd.DataFrame(regression_data_cs)
regression_df_ts = pd.DataFrame(regression_data_ts)

print(f"Cross-sectional regressions: {len(regression_df_cs)} variable-year combinations")
print(f"Time-series regressions: {len(regression_df_ts)} variable-region combinations")
regression_df_cs.head()

Cross-sectional regressions: 315 variable-year combinations
Time-series regressions: 441 variable-region combinations


,TIME_PERIOD,Variable,slope,intercept,r_squared,p_value,regression_type
0,2010,Difficult_Parking_V,18.206901,368.274573,0.187530,0.056488,cross_sectional
1,2010,Difficult_Parking_VQ,10.419881,294.441355,0.140752,0.103111,cross_sectional
2,2010,Poor_Public_Transport_V,-40.531609,1066.916947,0.174697,0.066678,cross_sectional
3,2010,Poor_Public_Transport_VQ,-12.994701,1035.115113,0.096776,0.181856,cross_sectional
4,2010,Poor_Road_Conditions_V,5.238608,556.733889,0.024099,0.513418,cross_sectional


In [148]:
# Create interactive scatter plot with regression line
# Regression type selector (cross-sectional vs time-series)
regression_type_dropdown = alt.binding_select(
    options=['Cross-sectional (choose Year and X-axis Variable)', 'Time-series (choose Region and X-axis Variable)'],
    name='Regression Type: '
)
regression_type_param = alt.param(
    name='regression_type_select',
    value='Cross-sectional (choose Year and X-axis Variable)',
    bind=regression_type_dropdown
)

# Year slider (for cross-sectional)
year_list = sorted([int(y) for y in data_for_regression['TIME_PERIOD'].unique()])
year_slider = alt.binding_range(
    min=min(year_list), 
    max=max(year_list), 
    step=1, 
    name='Year: '
)
year_param = alt.param(name='year_select', value=2024, bind=year_slider)

# Region dropdown (for time-series)
# Put 'All Regions' first, then sort the rest alphabetically
region_list_raw = data_for_regression['Territory'].unique().tolist()
region_list_sorted = sorted([r for r in region_list_raw if r != 'All Regions'])
region_list = ['All Regions'] + region_list_sorted
region_dropdown = alt.binding_select(options=region_list, name='Select Region: ')
region_param = alt.param(name='region_select', value='All Regions', bind=region_dropdown)

# Variable dropdown
variable_dropdown = alt.binding_select(options=x_variables, name='X-axis Variable: ')
variable_param = alt.param(name='variable_select', value='Traffic_Problems_VQ', bind=variable_dropdown)

# Prepare data for cross-sectional visualization
data_melted_cs = data_for_regression.melt(
    id_vars=['Territory', 'TIME_PERIOD', 'Accidents_Per_100k', 'Population', 'Total_Accidents'],
    value_vars=x_variables,
    var_name='Variable',
    value_name='X_Value'
)
data_melted_cs['X_Value'] = pd.to_numeric(data_melted_cs['X_Value'], errors='coerce')
data_melted_cs = data_melted_cs.merge(
    regression_df_cs[['TIME_PERIOD', 'Variable', 'slope', 'intercept', 'r_squared', 'p_value']],
    on=['TIME_PERIOD', 'Variable'],
    how='left'
)
data_melted_cs['Y_Predicted'] = data_melted_cs['slope'] * data_melted_cs['X_Value'] + data_melted_cs['intercept']
data_melted_cs['regression_type'] = 'Cross-sectional (choose Year and X-axis Variable)'

# Prepare data for time-series visualization
data_melted_ts = data_for_regression.melt(
    id_vars=['Territory', 'TIME_PERIOD', 'Accidents_Per_100k', 'Population', 'Total_Accidents'],
    value_vars=x_variables,
    var_name='Variable',
    value_name='X_Value'
)
data_melted_ts['X_Value'] = pd.to_numeric(data_melted_ts['X_Value'], errors='coerce')
data_melted_ts = data_melted_ts.merge(
    regression_df_ts[['Territory', 'Variable', 'slope', 'intercept', 'r_squared', 'p_value']],
    on=['Territory', 'Variable'],
    how='left'
)
data_melted_ts['Y_Predicted'] = data_melted_ts['slope'] * data_melted_ts['X_Value'] + data_melted_ts['intercept']
data_melted_ts['regression_type'] = 'Time-series (choose Region and X-axis Variable)'

# Combine both datasets
data_melted_combined = pd.concat([data_melted_cs, data_melted_ts], ignore_index=True)

# Create regression equation and stats text
data_melted_combined['equation'] = (
    'y = ' + data_melted_combined['slope'].round(2).astype(str) + 'x + ' + 
    data_melted_combined['intercept'].round(2).astype(str)
)
data_melted_combined['stats_text'] = (
    'R² = ' + data_melted_combined['r_squared'].round(4).astype(str) + 
    ', p = ' + data_melted_combined['p_value'].round(4).astype(str)
)

print(f"Prepared {len(data_melted_combined)} rows for visualization")
print(f"Years: {year_list}")
print(f"Regions: {len(region_list)}")
data_melted_combined.head()

Prepared 13230 rows for visualization
Years: [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
Regions: 21


,Territory,TIME_PERIOD,Accidents_Per_100k,Population,Total_Accidents,Variable,X_Value,slope,intercept,r_squared,p_value,Y_Predicted,regression_type,equation,stats_text
0,All Regions,2010,703.405888,59816673,420754,Difficult_Parking_V,18.7,18.206901,368.274573,0.187530,0.056488,708.743630,Cross-sectional (choose Year and X-axis Variable),y = 18.21x + 368.27,"R² = 0.1875, p = 0.0565"
1,All Regions,2011,677.560251,59816673,405294,Difficult_Parking_V,17.9,15.770646,392.582381,0.109524,0.154081,674.876940,Cross-sectional (choose Year and X-axis Variable),y = 15.77x + 392.58,"R² = 0.1095, p = 0.1541"
2,All Regions,2012,618.436268,59816673,369928,Difficult_Parking_V,14.8,26.802258,221.902862,0.209638,0.042352,618.576286,Cross-sectional (choose Year and X-axis Variable),y = 26.8x + 221.9,"R² = 0.2096, p = 0.0424"
3,All Regions,2013,596.793473,59816673,356982,Difficult_Parking_V,15.8,17.954222,332.038048,0.197898,0.049371,615.714754,Cross-sectional (choose Year and X-axis Variable),y = 17.95x + 332.04,"R² = 0.1979, p = 0.0494"
4,All Regions,2014,581.874555,59816673,348058,Difficult_Parking_V,14.9,22.774043,255.787409,0.236681,0.029618,595.120646,Cross-sectional (choose Year and X-axis Variable),y = 22.77x + 255.79,"R² = 0.2367, p = 0.0296"


In [149]:
# Create the scatter plot with regression line
scatter_regression = alt.Chart(data_melted_combined).mark_point(
    filled=True,
    size=60,
    opacity=1.0,
    color='royalblue'
).encode(
    x=alt.X('X_Value:Q', 
            title='Selected Variable Value',
            scale=alt.Scale(zero=False)),
    y=alt.Y('Accidents_Per_100k:Q', 
            title='Accidents per 100,000 inhabitants',
            scale=alt.Scale(zero=False)),
    tooltip=[
        alt.Tooltip('Territory:N', title='Region'),
        alt.Tooltip('TIME_PERIOD:Q', title='Year'),
        alt.Tooltip('Population:Q', title='Population', format=',.0f'),
        alt.Tooltip('Total_Accidents:Q', title='Total Accidents', format=',.0f'),
        alt.Tooltip('Accidents_Per_100k:Q', title='Accidents per 100k', format='.2f'),
        alt.Tooltip('X_Value:Q', title='X Value', format='.2f'),
        alt.Tooltip('Variable:N', title='Variable')
    ]
).add_params(
    regression_type_param,
    year_param,
    region_param,
    variable_param
).transform_filter(
    alt.datum.Variable == variable_param
).transform_filter(
    # Filter based on regression type
    (
        (alt.datum.regression_type == regression_type_param) &
        (
            # If cross-sectional, filter by year (exclude 'All Regions')
            ((regression_type_param == 'Cross-sectional (choose Year and X-axis Variable)') & 
             (alt.datum.TIME_PERIOD == year_param) & 
             (alt.datum.Territory != 'All Regions')) |
            # If time-series, filter by region (include 'All Regions')
            ((regression_type_param == 'Time-series (choose Region and X-axis Variable)') & (alt.datum.Territory == region_param))
        )
    )
)

# Create the regression line
regression_line = alt.Chart(data_melted_combined).mark_line(
    color='red',
    size=3
).encode(
    x='X_Value:Q',
    y='Y_Predicted:Q',
    tooltip=[
        alt.Tooltip('equation:N', title='Equation'),
        alt.Tooltip('r_squared:Q', title='R²', format='.4f'),
        alt.Tooltip('p_value:Q', title='p-value', format='.4f')
    ]
).transform_filter(
    alt.datum.Variable == variable_param
).transform_filter(
    (
        (alt.datum.regression_type == regression_type_param) &
        (
            # If cross-sectional, filter by year (exclude 'All Regions')
            ((regression_type_param == 'Cross-sectional (choose Year and X-axis Variable)') & 
             (alt.datum.TIME_PERIOD == year_param) & 
             (alt.datum.Territory != 'All Regions')) |
            # If time-series, filter by region (include 'All Regions')
            ((regression_type_param == 'Time-series (choose Region and X-axis Variable)') & (alt.datum.Territory == region_param))
        )
    )
)

# Create text annotation for regression statistics
regression_text = alt.Chart(data_melted_combined).mark_text(
    align='left',
    baseline='top',
    dx=5,
    dy=5,
    fontSize=12,
    fontWeight='bold'
).encode(
    x=alt.value(10),  # Fixed position
    y=alt.value(10),  # Fixed position
    text='equation:N'
).transform_filter(
    alt.datum.Variable == variable_param
).transform_filter(
    (
        (alt.datum.regression_type == regression_type_param) &
        (
            # If cross-sectional, filter by year (exclude 'All Regions')
            ((regression_type_param == 'Cross-sectional (choose Year and X-axis Variable)') & 
             (alt.datum.TIME_PERIOD == year_param) & 
             (alt.datum.Territory != 'All Regions')) |
            # If time-series, filter by region (include 'All Regions')
            ((regression_type_param == 'Time-series (choose Region and X-axis Variable)') & (alt.datum.Territory == region_param))
        )
    )
).transform_aggregate(
    equation='min(equation)',
    groupby=['regression_type', 'TIME_PERIOD', 'Territory', 'Variable']
)

# Create text annotation for R² and p-value
stats_text = alt.Chart(data_melted_combined).mark_text(
    align='left',
    baseline='top',
    dx=5,
    dy=25,
    fontSize=11
).encode(
    x=alt.value(10),
    y=alt.value(10),
    text='stats_text:N'
).transform_filter(
    alt.datum.Variable == variable_param
).transform_filter(
    (
        (alt.datum.regression_type == regression_type_param) &
        (
            # If cross-sectional, filter by year (exclude 'All Regions')
            ((regression_type_param == 'Cross-sectional (choose Year and X-axis Variable)') & 
             (alt.datum.TIME_PERIOD == year_param) & 
             (alt.datum.Territory != 'All Regions')) |
            # If time-series, filter by region (include 'All Regions')
            ((regression_type_param == 'Time-series (choose Region and X-axis Variable)') & (alt.datum.Territory == region_param))
        )
    )
).transform_aggregate(
    stats_text='min(stats_text)',
    groupby=['regression_type', 'TIME_PERIOD', 'Territory', 'Variable']
)

# Combine all layers
chart = (scatter_regression + regression_line + regression_text + stats_text).properties(
    width=700,
    height=500,
    title='Relationship between Mobility/Commute Factors and Traffic Accidents per 100k'
)

chart

alt.LayerChart(...)